# Slice Drift Tutorial

A regression that only affects one cohort is easy to miss when you look at a
single global number. This notebook checks a numeric column per cohort, then
corrects for the fact that checking several cohorts means running several
tests.

`SliceDriftDetector` used to wrap this loop, but it was removed along with the
other flat detectors. The loop is short enough to write directly, and doing so
keeps the multiple-testing step visible rather than hidden inside a class.

In [ ]:
import numpy as np
import pandas as pd

from drift_control import UnifiedDriftDetector, adjust_pvalues

## Data

Cohort `A` is unchanged between the two samples. Cohort `B` shifts from a mean
of 100 to 125 and widens.

In [ ]:
rng = np.random.default_rng(123)

baseline = pd.DataFrame(
    {
        "cohort": ["A"] * 400 + ["B"] * 400,
        "amount": np.concatenate(
            [rng.normal(100, 15, size=400), rng.normal(100, 15, size=400)]
        ),
    }
)

current = pd.DataFrame(
    {
        "cohort": ["A"] * 400 + ["B"] * 400,
        "amount": np.concatenate(
            [rng.normal(100, 15, size=400), rng.normal(125, 20, size=400)]
        ),
    }
)

baseline.head()

## The global check

With a shift this large the global test does fire, so this is not a case of the
aggregate hiding the problem outright. What the global result cannot tell you
is *which* cohort moved, which is what you need before you can act on it.

In [ ]:
detector = UnifiedDriftDetector(method="ks", alpha=0.05)

overall = detector.detect_drift(
    baseline["amount"].to_numpy(),
    current["amount"].to_numpy(),
)
print(f"drift={overall.drift}  p={overall.p_value:.3g}")

## Per cohort

Run the same detector on each cohort separately. Cohorts too small to say
anything about are skipped rather than reported as clean, since a test with no
power is not evidence of no drift.

Testing several cohorts inflates the chance of at least one false positive, so
the p-values are adjusted with Benjamini-Hochberg before they are read.

In [ ]:
MIN_SAMPLES = 20

rows = []
for cohort in sorted(set(baseline["cohort"]) | set(current["cohort"])):
    ref = baseline.loc[baseline["cohort"] == cohort, "amount"].to_numpy()
    cur = current.loc[current["cohort"] == cohort, "amount"].to_numpy()
    if len(ref) < MIN_SAMPLES or len(cur) < MIN_SAMPLES:
        continue

    result = detector.detect_drift(ref, cur)
    rows.append(
        {
            "cohort": cohort,
            "n_reference": len(ref),
            "n_current": len(cur),
            "p_value": result.p_value,
        }
    )

by_cohort = pd.DataFrame(rows)
by_cohort["p_adjusted"] = adjust_pvalues(by_cohort["p_value"].tolist(), method="bh")
by_cohort["drift"] = by_cohort["p_adjusted"] < 0.05
by_cohort

## Interpretation

Cohort `B` comes back with an adjusted p-value far below the threshold and `A`
does not, which localises the regression to `B`. That is the part the global
check cannot give you.

Worth keeping in mind: the more cohorts you slice by, the more tests you run,
and the adjustment gets correspondingly stricter. Slicing by a high-cardinality
column will cost you power on every cohort, so slice by something you would
actually act on.